# WEEK 3 - Homework 3: Classification

> **Note**: this homework uses the pinned 2026 lead-scoring release (`cohorts/2026/data/course_lead_scoring_2026.csv`).

The target for the classification task is the `converted` variable - has the client signed up to the platform or not.

In [90]:
import numpy as np
import pandas as pd

df = pd.read_csv("../cohorts/2026/data/course_lead_scoring_2026.csv")
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


## Data preparation

- Check if the missing values are presented in the features.
- If there are missing values:
  - For categorical features, replace them with `'NA'`
  - For numerical features, replace with `0.0`

In [91]:
"""
for col in df.columns:
    if col != 'converted':
        if df[col].dtype == 'object' or df[col].dtype == 'str':
            print(f"Filling missing values in column '{col}' with 'NA'")
            df[col] = df[col].fillna('NA')
        else:
            df[col] = df[col].fillna(0.0)
            print(f"Filling missing values in column '{col}' with 0.0")

print(df.isnull().sum())
"""

'\nfor col in df.columns:\n    if col != \'converted\':\n        if df[col].dtype == \'object\' or df[col].dtype == \'str\':\n            print(f"Filling missing values in column \'{col}\' with \'NA\'")\n            df[col] = df[col].fillna(\'NA\')\n        else:\n            df[col] = df[col].fillna(0.0)\n            print(f"Filling missing values in column \'{col}\' with 0.0")\n\nprint(df.isnull().sum())\n'

In [92]:
categorical_columns = df.select_dtypes(include=['object', 'string']).columns.tolist()
numerical_columns = [c for c in df.select_dtypes(include='number').columns if c != 'converted']
print(df.isnull().sum())

df[categorical_columns] = df[categorical_columns].fillna('NA')
df[numerical_columns] = df[numerical_columns].fillna(0.0)
print(df.isnull().sum())

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64
lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64


## Question 1

What is the most frequent observation (mode) for the column `industry`?

- `NA`
- `technology`
- `healthcare`
- `retail`

In [93]:
df.industry.mode()

0    technology
Name: industry, dtype: str

## Question 2

Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

- `interaction_count` and `lead_score`
- `number_of_courses_viewed` and `lead_score`
- `number_of_courses_viewed` and `interaction_count`
- `annual_income` and `interaction_count`

Only consider the pairs above when answering this question.

In [94]:
df[numerical_columns].corr()


,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


## Split the data

Split the data with these exact calls, and make sure that the target value `converted` is not in your dataframe.

In [95]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values 
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

## Question 3

- Calculate the mutual information score between `converted` and other categorical variables in the dataset. Use the training set only.
- Round the scores to 2 decimals using `round(score, 2)`.

Which of these variables has the biggest mutual information score?

- `industry`
- `location`
- `lead_source`
- `employment_status`

In [96]:
from sklearn.metrics import mutual_info_score

def mutual_info_converted_score(series):
    return mutual_info_score(series, y_train)

mi = df_train[categorical_columns].apply(mutual_info_converted_score)
round(mi.sort_values(ascending=False), 2)


lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

## Question 4

- Now let's train a logistic regression.
- Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
- Fit the model on the training dataset.
  - To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
  - `model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)`
- Calculate the accuracy on the validation dataset and round it to 2 decimal digits.

What accuracy did you get?

- 0.55
- 0.65
- 0.75
- 0.85

In [97]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical_columns + numerical_columns].to_dict(orient="records")
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical_columns + numerical_columns].to_dict(orient="records")
X_val = dv.transform(val_dict)

# print(dv.get_feature_names_out())

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# print(model.intercept_[0])    
# print(model.coef_[0])            

y_pred = model.predict_proba(X_val)[:, 1]   
converted_decision = (y_pred >= 0.5)
accuracy = (y_val == converted_decision).mean()

accuracy

np.float64(0.645)

## Question 5

- Let's find the least useful feature using the *feature elimination* technique.
- Train a model using the same features and parameters as in Q4 (without rounding).
- Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
- For each feature, calculate the difference between the original accuracy and the accuracy without the feature.

Which of following feature has the smallest difference?

- `'lead_source'`
- `'number_of_courses_viewed'`
- `'interaction_count'`

> **Note**: The difference doesn't have to be positive.

In [98]:
features = categorical_columns + numerical_columns
original_accuracy = accuracy

for feature in features:
  subset = [f for f in features if f != feature]

  dv = DictVectorizer(sparse=False)

  train_dict = df_train[subset].to_dict(orient="records")
  X_train = dv.fit_transform(train_dict)

  val_dict = df_val[subset].to_dict(orient="records")
  X_val = dv.transform(val_dict)

  model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
  model.fit(X_train, y_train)          

  y_pred = model.predict_proba(X_val)[:, 1]   
  converted_decision = (y_pred >= 0.5)
  accuracy = (y_val == converted_decision).mean()

  print(f"{feature:<20} {accuracy:<10.3f} {original_accuracy - accuracy:<10.3f}")

lead_source          0.642      0.003     
industry             0.645      0.000     
employment_status    0.645      0.000     
location             0.645      0.000     
annual_income        0.724      -0.079    
number_of_courses_viewed 0.643      0.002     
interaction_count    0.601      0.044     
lead_score           0.644      0.001     


## Question 6

- Now let's train a regularized logistic regression.
- Let's try the following values of the parameter `C`: `[0.000001, 0.00001, 0.0001, 0.001]`.
- Train models using all the features as in Q4.
- Calculate the accuracy on the validation dataset and round it to 3 decimal digits.

Which of these `C` leads to the best accuracy on the validation set?

- 0.000001
- 0.00001
- 0.0001
- 0.001

> **Note**: If there are multiple options, select the smallest `C`.

In [100]:
cs = [0.000001, 0.00001, 0.0001, 0.001]

dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical_columns + numerical_columns].to_dict(orient="records")
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical_columns + numerical_columns].to_dict(orient="records")
X_val = dv.transform(val_dict)

for c in cs:
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)          

    y_pred = model.predict_proba(X_val)[:, 1]   
    converted_decision = (y_pred >= 0.5)
    accuracy = (y_val == converted_decision).mean()

    print(f"{c:<20} {accuracy:<10.3f}")

1e-06                0.598     
1e-05                0.598     
0.0001               0.613     
0.001                0.645     


## Submit the results

https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw03